# Information Retrieval in the 1914 French Press

## Fetching the Data

In [ ]:
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/ghsparse.sh -O /tmp/ghsparse.sh && bash /tmp/ghsparse.sh french-press-1914

In [ ]:
!ls french-press-1914/le_figaro/1914/

## Corpus

This corpus is made of articles from the French press of 1914.

One folder per newspaper, each containing one JSON file per publication date.

The JSON files contain several fields, three of which we'll use in the rest of this lab:

- `contentAsText`, a list of strings (paragraphs are delimited in these strings by line breaks — `\n`)
- `date`, the newspaper's publication date, represented by a list containing one string (if the `content` variable holds the loaded JSON file, access it with `content["date"][0]`)
- `title`, the newspaper's name, represented by a list containing one string (if the `content` variable holds the loaded JSON file, access it with `content["title"][0]`)

Implement a function that takes the corpus path as input and returns a 3-column `numpy.ndarray`:

- the first column contains the paragraphs of the corpus editions (so one row per paragraph of the corpus), meeting the following conditions:
  - only paragraphs of more than 10 words
  - lowercase
  - without punctuation
- the second column contains the newspaper the paragraph comes from
- the third column contains the date of the edition the paragraph comes from

In [ ]:
# Your code here

### Solution

In [ ]:
import datetime
import json
import pathlib
import re

import numpy
import tqdm.notebook


def preprocess_text(text: str) -> str:
  out = re.sub(r"\W+", " ", text)
  return out.lower()


def get_data(directory: pathlib.Path) -> numpy.ndarray:
  data = []
  for edition in tqdm.notebook.tqdm(list(directory.rglob("*.json"))):
    with open(edition, encoding="utf8") as fh:
      content = json.load(fh)
    newspaper = content["title"][0]
    date = datetime.date.fromisoformat(content["date"][0]).strftime("%d/%m/%Y")
    for item in content["contentAsText"]:
      for paragraph in item.split("\n"):
        processed_paragraph = preprocess_text(paragraph)
        if len(processed_paragraph.split()) > 10:
          data.append((processed_paragraph, newspaper, date))
  return numpy.array(data)


data = get_data(pathlib.Path("french-press-1914"))

## TF-IDF Representation

Use a [`sklearn.feature_extraction.text.TfidfVectorizer`](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html) to build the matrix `X` of the corpus paragraphs turned into TF-IDF weight vectors.

In [ ]:
# Your code here

### Solution

In [ ]:
import sklearn.feature_extraction.text

vectorizer = sklearn.feature_extraction.text.TfidfVectorizer()
X = vectorizer.fit_transform(data[:, 0])

## Similarity

Implement a function `best_score(query: str) -> None` that:

- takes a query string as input
- displays the newspapers and dates of the 5 best results in terms of cosine similarity

In [ ]:
# Your code here

### Solution

In [ ]:
import sklearn.metrics.pairwise


def best_score(query: str) -> None:
  query_vector = vectorizer.transform([query.lower()])
  cosine = sklearn.metrics.pairwise.cosine_similarity(X, query_vector)[:, 0]
  # We use the opposite of the cosine array in the argsort call to get the
  # indices of the array sorted by decreasing values, before keeping the
  # first 5
  top5 = (-cosine).argsort()[:5]
  for index, (text, newspaper, date), similarity in zip(top5, data[top5], cosine[top5]):
    print(
      f"{newspaper:10} on {date}; similarity {similarity:5.2f}; index {index:6}; {text}"
    )

## BM25

First try `best_score("guerre allemands")`: what do you notice about the length of the top paragraphs? The corpus is in French, so the queries stay in French.

The course presents Okapi BM25, an improvement on the TF-IDF dot product, which compares each document's length to the average length:

$$
\text{score}(D, Q)
= \sum_{i = 1}^n
\text{IDF}(q_i)
\cdot
\frac{f(q_i, D) \cdot (k_1 + 1)}
     {f(q_i, D) + k_1 \cdot \left(1 - b + b \cdot \frac{|D|}{\bar{|D|}}\right)}
$$

where:

- $q_1, \dots, q_n$ are the terms of the query $Q$
- $\text{IDF}(q_i) = \log\frac{N}{df_i}$, the one from TF-IDF: $df_i$ of the base's $N$ documents contain $q_i$
- $f(q_i, D)$ is the frequency of the term $q_i$ in the document $D$
- $|D|$ is the number of terms in $D$, and $\bar{|D|}$ the average length of the base's documents
- $k_1 = 1.5$ and $b = 0.75$

Here, a document is a paragraph. Count the terms of each paragraph with a `CountVectorizer`, in a variable `counter` and a matrix `counts`, and keep each paragraph's length in `lengths`. Then implement the function `bm25_scores`, which returns each paragraph's score for a query.

**Useful documentation**

- [`sklearn.feature_extraction.text.CountVectorizer`](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html)
- [`scipy.sparse.csr_matrix.getnnz`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.sparse.csr_matrix.getnnz.html)
- [`numpy.log`](https://numpy.org/doc/stable/reference/generated/numpy.log.html)

In [ ]:
# Your code here


def bm25_scores(query: str, k1: float = 1.5, b: float = 0.75) -> numpy.ndarray:
  # Your code here
  ...

### Additional hints

The calls and parameters to use:

- `counter.fit_transform(data[:, 0])`: the sparse matrix `counts`, with one row per paragraph and one column per term
- `counts.getnnz(axis=0)`: for each term, the number of paragraphs containing it ($df_i$)
- `numpy.asarray(counts.sum(axis=1)).ravel()`: for each paragraph, its number of terms ($|D|$). `CountVectorizer` ignores one-letter words, so 12 paragraphs have no term at all: count them as 1 (`numpy.maximum(..., 1)`), or $b = 1$ divides by zero
- `counter.transform([query]).indices`: the columns of the query's terms. `CountVectorizer` lowercases it, splits it and ignores the words absent from the corpus
- `counts[:, term].toarray().ravel()`: a term's column, that is $f(q_i, D)$ for all the paragraphs. numpy's operations then apply to every paragraph at once, with no loop over the documents
- to check: `N` is 48,333, `lengths.mean()` about 178 and `bm25_scores("jaurès assassiné").max()` about 13.51

A skeleton to copy into the exercise cell, replacing each `...`:

```python
counter = sklearn.feature_extraction.text.CountVectorizer()
counts = ...
N = counts.shape[0]
df = ...  # df_i for each term
lengths = numpy.maximum(..., 1)  # |D| for each paragraph


def bm25_scores(query: str, k1: float = 1.5, b: float = 0.75) -> numpy.ndarray:
  # The part of the denominator that depends only on the paragraph
  normalization = k1 * (1 - b + b * ...)
  scores = numpy.zeros(N)
  for term in counter.transform([query]).indices:
    f = ...  # f(q_i, D) for all the paragraphs
    idf = ...
    scores += ...
  return scores
```

### Solution

In [ ]:
best_score("guerre allemands")

counter = sklearn.feature_extraction.text.CountVectorizer()
counts = counter.fit_transform(data[:, 0])
N = counts.shape[0]
df = counts.getnnz(axis=0)
# CountVectorizer ignores one-letter words: 12 paragraphs have no term at all, and we
# count them as 1 so as not to divide by zero when b is 1
lengths = numpy.maximum(numpy.asarray(counts.sum(axis=1)).ravel(), 1)


def bm25_scores(query: str, k1: float = 1.5, b: float = 0.75) -> numpy.ndarray:
  normalization = k1 * (1 - b + b * lengths / lengths.mean())
  scores = numpy.zeros(N)
  for term in counter.transform([query]).indices:
    f = counts[:, term].toarray().ravel()
    idf = numpy.log(N / df[term])
    scores += idf * f * (k1 + 1) / (f + normalization)
  return scores


print(bm25_scores("jaurès assassiné").max())

## Comparison with the cosine

Implement a function `compare(query: str) -> None` that displays, for the cosine similarity and then for BM25, the 5 best results: the score, the length, the index and the start of the text. A result's whole text is in `data[index, 0]`.

Then look at:

1. on `guerre allemands` (Germans war) and `jaurès assassiné` (Jaurès assassinated), the length of the paragraphs each method ranks first, against the corpus's median length. Which one brings up the article on Jaurès's assassination?
2. the length of BM25's 5 best results on `guerre allemands`, with $b$ at 0, then 0.75, then 1: what does $b$ control?
3. on `mobilisation générale` (general mobilization), the results of both methods: how many really announce a general mobilization? Why?

Conclude: what does length normalization bring, and what does it leave unsolved?

**Useful documentation**

- [`numpy.argsort`](https://numpy.org/doc/stable/reference/generated/numpy.argsort.html)
- [`numpy.median`](https://numpy.org/doc/stable/reference/generated/numpy.median.html)

In [ ]:
def compare(query: str) -> None:
  # Your code here
  ...


# Your code here

### Additional hints

The calls and parameters to use:

- `(-scores).argsort()[:5]`: the indices of the 5 best scores, as in `best_score`
- `sklearn.metrics.pairwise.cosine_similarity(X, query_vector)[:, 0]`: each paragraph's similarity with the query
- `data[index, 0][:80]`: the start of a paragraph
- `bm25_scores("guerre allemands", b=b)`: BM25 with another value of $b$
- to check: `numpy.median(lengths)` is 102

A skeleton to copy into the exercise cell, replacing each `...`:

```python
def show_top5(scores: numpy.ndarray) -> None:
  for index in ...:  # the indices of the 5 best scores
    start = data[index, 0][:80]
    print(f"{scores[index]:5.2f}; {lengths[index]:4} terms; {index:6}; {start}")


def compare(query: str) -> None:
  print(f"Query: {query}")
  print("Cosine")
  query_vector = vectorizer.transform([query])
  show_top5(...)  # each paragraph's similarity with the query
  print("BM25")
  show_top5(...)


print("Median length:", ...)
compare("guerre allemands")
compare("jaurès assassiné")

for b in (0, 0.75, 1):
  best = ...  # the indices of BM25's 5 best scores with this b
  print(f"b = {b}: {lengths[best]}")

compare("mobilisation générale")
```

### Solution

In [ ]:
def show_top5(scores: numpy.ndarray) -> None:
  for index in (-scores).argsort()[:5]:
    start = data[index, 0][:80]
    print(f"{scores[index]:5.2f}; {lengths[index]:4} terms; {index:6}; {start}")


def compare(query: str) -> None:
  print(f"Query: {query}")
  print("Cosine")
  query_vector = vectorizer.transform([query])
  show_top5(sklearn.metrics.pairwise.cosine_similarity(X, query_vector)[:, 0])
  print("BM25")
  show_top5(bm25_scores(query))


print("Median length:", numpy.median(lengths))
compare("guerre allemands")
compare("jaurès assassiné")

for b in (0, 0.75, 1):
  best = (-bm25_scores("guerre allemands", b=b)).argsort()[:5]
  print(f"b = {b}: {lengths[best]}")

compare("mobilisation générale")

What these results show:

- The cosine similarity divides by the norm of the paragraph's vector: in a headline, the query's terms weigh heavily. On `guerre allemands`, its 5 best results have 12 to 14 terms, against a median of 102. On `jaurès assassiné`, it ranks a wine merchant first (10 terms), and the article on Jaurès (691 terms) only comes 3rd.
- BM25 ranks that article first, and the next four are also about Jaurès's death. On `guerre allemands`, two paragraphs of 84 and 85 terms enter the top 5, but three headlines stay: short texts are less favored, not ruled out.
- $b$ controls this normalization: at 0, the top 5 have 656 to 1,042 terms, because a long text contains the terms more often; at 1, BM25 goes back to headlines of 12 to 14 terms, like the cosine.
- Normalization doesn't solve everything: on `mobilisation générale`, only one result out of 5 really announces one (the Netherlands'), in both methods. A general strike in South Africa, with a mobilization of troops, comes 1st with the cosine and 2nd with BM25: a bag of words doesn't know that the two terms must be adjacent. The account of the order being posted in Paris only comes 6th with BM25, 16th with the cosine.

Three queries read by hand measure nothing: comparing two engines takes relevance judgments.